In [1]:
# %% [markdown]
# # Download & Clip DEM (COP30) for Central Asia countries (robuste)
#
# - Input: One AOI (shapefile, gpkg, etc.) per country
# - Output: One DEM GeoTIFF per country, clipped to AOI
#
# Features:
#   - Uses COP30 (Copernicus GLO-30 DEM) from OpenTopography
#   - Splits bbox into tiles (3° x 3°)
#   - Downloads each tile, then IMMEDIATELY checks it with rasterio
#   - Retries up to 3 times if a tile is corrupted
#   - Skips bad tiles in mosaic; fails only if 0 valid tiles
#
# Requirements:
#   conda install -c conda-forge geopandas rasterio shapely requests tqdm numpy
#   or:
#   pip install geopandas rasterio shapely requests tqdm numpy

# %%
import os
from pathlib import Path

import geopandas as gpd
import numpy as np
import requests
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask
from shapely.geometry import mapping
from tqdm.auto import tqdm

# %% [markdown]
# ## 1. Configuration

# %%
# 🔑 1. OpenTopography API key (MET LA TIENTE ICI EN LOCAL, PAS DANS GIT)
OPENTOPO_API_KEY = "431c0d11ec136fefaa6f05a4cfb62895"

# 🌍 2. DEM type
DEMTYPE = "FABDEM"   # tu peux mettre "SRTMGL1" plus tard si tu veux tester SRTM

# 📁 3. Output root dir (adapter au chemin de ton repo)
OUTPUT_ROOT = Path("../data/DEM")

# 📁 4. AOI paths per country (ADAPTE si besoin)
AOI_CONFIG = {
  # "kaz": {
   #    "label": "Kazakhstan",
#       "aoi_path": "../data/kazak/gadm41_KAZ_0.shp",
  # },
 "uzb": {
         "label": "Uzbekistan",
      "aoi_path": "../data/uzbek/uzb_admbnda_adm0_2018b.shp",
  },
   #   "tkm": {
   #       "label": "Turkmenistan",
  #        "aoi_path": "../data/turkmini/gadm41_TKM_0.shp",
  #    },
 #   "kgz": {
 #       "label": "Kyrgyzstan",
 #       "aoi_path": "../data/kyrgy/gadm41_KGZ_0.shp",
  #  },
  #  "tjk": {
 #       "label": "Tajikistan",
 #       "aoi_path": "../data/tajik/aoi.shp",
  #  },
}

print("DEM type:", DEMTYPE)
print("AOI config:")
for code, cfg in AOI_CONFIG.items():
    print(f"  {code}: {cfg['aoi_path']}")

if OPENTOPO_API_KEY == "METTRE_TA_CLE_ICI":
    raise ValueError("⚠️ Tu dois mettre ta clé OpenTopography dans OPENTOPO_API_KEY avant de lancer.")


# %% [markdown]
# ## 2. OpenTopography: download DEM for a bbox (avec retry & validation)

# %%
OPENTOPO_BASE_URL = "https://portal.opentopography.org/API/globaldem"


def _is_valid_geotiff(path):
    """
    Essaie d'ouvrir le GeoTIFF avec rasterio.
    Retourne True si ça marche, False si erreur.
    """
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with rasterio.open(path) as src:
            _ = src.read(1)
        return True
    except Exception as e:
        print(f"       ⚠️ rasterio can't read {path.name}: {e}")
        return False


def download_dem_bbox(demtype, west, east, south, north, out_path, api_key, max_retries=3):
    """
    Download a DEM from OpenTopography for a given bounding box (W/E/S/N, EPSG:4326).
    Save GeoTIFF at out_path.
    Retries up to max_retries times if file is corrupted / unreadable.
    """
    params = {
        "demtype": demtype,
        "west": west,
        "east": east,
        "south": south,
        "north": north,
        "outputFormat": "GTiff",
        "API_Key": api_key,
    }

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    for attempt in range(1, max_retries + 1):
        print(
            f"    -> Requesting {demtype} for bbox: "
            f"W={west:.4f}, E={east:.4f}, S={south:.4f}, N={north:.4f} "
            f"(attempt {attempt}/{max_retries})"
        )

        tmp_path = out_path.with_suffix(f".tmp_{attempt}.tif")

        try:
            with requests.get(OPENTOPO_BASE_URL, params=params, stream=True, timeout=120) as r:
                if r.status_code != 200:
                    try:
                        preview = r.text[:500]
                    except Exception:
                        preview = "<binary response>"
                    raise RuntimeError(
                        f"HTTP {r.status_code} from OpenTopography.\n"
                        f"URL: {r.url}\n"
                        f"Body preview:\n{preview}"
                    )

                with open(tmp_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)

            # Validation rapide
            if _is_valid_geotiff(tmp_path):
                tmp_size_mb = tmp_path.stat().st_size / (1024 * 1024)
                print(f"       ✅ tile OK ({tmp_size_mb:.2f} MB) -> {out_path.name}")
                # On renomme le fichier valide vers le chemin final
                tmp_path.replace(out_path)
                return out_path
            else:
                print(f"       ❌ Downloaded file {tmp_path.name} is not a valid GeoTIFF, retrying...")
                tmp_path.unlink(missing_ok=True)

        except Exception as e:
            print(f"       ❌ Error during download attempt {attempt}: {e}")
            tmp_path.unlink(missing_ok=True)

    # Si on arrive ici, tous les essais ont échoué
    print(f"       ❌ Failed to download a valid DEM tile after {max_retries} attempts for bbox.")
    return None


# %% [markdown]
# ## 3. AOI loading & clipping

# %%
def load_aoi_geometry(aoi_path):
    """
    Load AOI file (shp, gpkg, etc.) and return a single geometry in EPSG:4326.
    If multiple features, dissolve them.
    """
    aoi_gdf = gpd.read_file(aoi_path)

    if len(aoi_gdf) > 1:
        aoi_gdf = aoi_gdf.dissolve()

    if aoi_gdf.crs is None:
        raise ValueError(f"AOI {aoi_path} has no CRS. Please set a CRS.")

    aoi_gdf = aoi_gdf.to_crs("EPSG:4326")
    geom = aoi_gdf.geometry.iloc[0]
    return geom


def clip_dem_to_aoi(raw_dem_path, geom, out_path, nodata_value=-32768):
    """
    Clip a DEM (raw_dem_path) to AOI geometry (shapely) and write GeoTIFF out_path.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(raw_dem_path) as src:
        src_nodata = src.nodata
        if src_nodata is not None:
            used_nodata = src_nodata
        else:
            used_nodata = nodata_value

        out_image, out_transform = mask(
            src,
            [mapping(geom)],
            crop=True,
            nodata=used_nodata,
        )

        out_meta = src.meta.copy()
        out_meta.update(
            {
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "nodata": used_nodata,
                "compress": "none",
            }
        )

    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(out_image)

    print(f"    -> Clipped DEM saved to: {out_path}")
    return out_path


# %% [markdown]
# ## 4. Generate tiles from AOI bounds

# %%
def generate_tiles_from_bounds(minx, miny, maxx, maxy, step_deg=3.0):
    """
    Split bounding box [minx, miny, maxx, maxy] into smaller tiles,
    each up to step_deg x step_deg in degrees.
    Returns a list of (west, east, south, north) bboxes.
    """
    tiles = []
    xs = np.arange(minx, maxx, step_deg)
    ys = np.arange(miny, maxy, step_deg)

    for x0 in xs:
        for y0 in ys:
            west = float(x0)
            east = float(min(x0 + step_deg, maxx))
            south = float(y0)
            north = float(min(y0 + step_deg, maxy))
            tiles.append((west, east, south, north))

    return tiles


# %% [markdown]
# ## 5. Mosaic tiles into a single DEM (en ignorant les tuiles cassées)

# %%
def mosaic_tiles(tile_paths, mosaic_path):
    """
    Mosaic a list of GeoTIFF tile paths into a single GeoTIFF.
    Tiles that rasterio cannot open are skipped.
    """
    mosaic_path = Path(mosaic_path)
    mosaic_path.parent.mkdir(parents=True, exist_ok=True)

    valid_srcs = []
    valid_paths = []

    for p in tile_paths:
        try:
            src = rasterio.open(p)
            # Test lecture rapide
            _ = src.read(1, window=((0, 1), (0, 1)))
            valid_srcs.append(src)
            valid_paths.append(p)
        except Exception as e:
            print(f"    ⚠️ Skipping invalid tile {p.name}: {e}")

    if not valid_srcs:
        raise ValueError("No valid tiles to mosaic (all failed to read).")

    mosaic_array, out_transform = merge(valid_srcs)
    out_meta = valid_srcs[0].meta.copy()
    for src in valid_srcs:
        src.close()

    out_meta.update(
        {
            "height": mosaic_array.shape[1],
            "width": mosaic_array.shape[2],
            "transform": out_transform,
            "compress": "none",
        }
    )

    with rasterio.open(mosaic_path, "w", **out_meta) as dst:
        dst.write(mosaic_array)

    print(f"    -> Mosaic DEM saved to: {mosaic_path} (from {len(valid_paths)} tiles)")
    return mosaic_path


# %% [markdown]
# ## 6. Process a single country

# %%
def process_country(code, label, aoi_path, demtype, api_key, output_root, step_deg=3.0):
    print("\n" + "=" * 80)
    print(f"Processing country: {label} ({code})")
    print(f"AOI file: {aoi_path}")

    aoi_path = Path(aoi_path)
    if not aoi_path.exists():
        print(f"⚠️ AOI file not found: {aoi_path}, skipping.")
        return

    # 1. AOI
    try:
        geom = load_aoi_geometry(aoi_path)
    except Exception as e:
        print(f"❌ Error loading AOI for {label}: {e}")
        return

    minx, miny, maxx, maxy = geom.bounds
    print(f"  AOI bounds (W, S, E, N): {minx:.4f}, {miny:.4f}, {maxx:.4f}, {maxy:.4f}")

    # 2. Tiles
    tiles = generate_tiles_from_bounds(minx, miny, maxx, maxy, step_deg=step_deg)
    print(f"  Number of tiles to download: {len(tiles)}")

    country_dir = output_root / code
    raw_tiles_dir = country_dir / "raw_tiles"
    raw_tiles_dir.mkdir(parents=True, exist_ok=True)

    tile_paths = []

    # 3. Download each tile
    for i, (west, east, south, north) in enumerate(tiles, start=1):
        tile_path = raw_tiles_dir / f"tile_{i:02d}.tif"
        if tile_path.exists() and _is_valid_geotiff(tile_path):
            print(f"    -> Tile {i:02d} already exists and is valid, skipping download.")
            tile_paths.append(tile_path)
            continue

        tile_result = download_dem_bbox(
            demtype=demtype,
            west=west,
            east=east,
            south=south,
            north=north,
            out_path=tile_path,
            api_key=api_key,
            max_retries=3,
        )

        if tile_result is not None and _is_valid_geotiff(tile_result):
            tile_paths.append(tile_result)
        else:
            print(f"    ⚠️ Tile {i:02d} could not be obtained as a valid GeoTIFF, skipping it.")

    if not tile_paths:
        print(f"❌ No valid tiles downloaded for {label}, skipping mosaic/clip.")
        return

    # 4. Mosaic
    mosaic_path = country_dir / f"dem_{code}_mosaic.tif"
    try:
        mosaic_dem_path = mosaic_tiles(tile_paths, mosaic_path)
    except Exception as e:
        print(f"❌ Error mosaicing tiles for {label}: {e}")
        return

    # 5. Clip
    clipped_dem_path = country_dir / f"dem_{code}.tif"
    try:
        clip_dem_to_aoi(mosaic_dem_path, geom, clipped_dem_path)
    except Exception as e:
        print(f"❌ Error clipping DEM for {label}: {e}")
        return

    if clipped_dem_path.exists():
        print(f"✅ Done for {label}. Final DEM: {clipped_dem_path}")
    else:
        print(f"⚠️ Something went wrong, {clipped_dem_path} was not created.")


# %% [markdown]
# ## 7. Run for all countries
# 💡 Pour tester, commence par UN SEUL pays (par ex. seulement "tkm" dans AOI_CONFIG),
# puis élargis à tous une fois que ça marche.

# %%
for code, cfg in tqdm(AOI_CONFIG.items(), desc="Countries"):
    process_country(
        code=code,
        label=cfg["label"],
        aoi_path=cfg["aoi_path"],
        demtype=DEMTYPE,
        api_key=OPENTOPO_API_KEY,
        output_root=OUTPUT_ROOT,
        step_deg=3.0,
    )

print("\n🎉 All done! Check your DEM files in:", OUTPUT_ROOT.resolve())

# %% [markdown]
# ## 8. (Optional) Quick check for one DEM in Python

# %%
# Exemple: vérifier Turkmenistan
test_path = OUTPUT_ROOT / "tkm" / "dem_tkm.tif"
if test_path.exists():
    print("Test DEM:", test_path)
    with rasterio.open(test_path) as src:
        arr = src.read(1).astype(float)
        nodata = src.nodata
        print("  CRS:", src.crs)
        print("  shape:", arr.shape)
        print("  nodata:", nodata)
        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)
        print("  min:", np.nanmin(arr))
        print("  max:", np.nanmax(arr))
else:
    print("No DEM file found for tkm at:", test_path)


DEM type: FABDEM
AOI config:
  uzb: ../data/uzbek/uzb_admbnda_adm0_2018b.shp


Countries:   0%|          | 0/1 [00:00<?, ?it/s]


Processing country: Uzbekistan (uzb)
AOI file: ../data/uzbek/uzb_admbnda_adm0_2018b.shp
  AOI bounds (W, S, E, N): 55.9966, 37.1843, 73.1323, 45.6052
  Number of tiles to download: 18
    -> Tile 01 already exists and is valid, skipping download.
    -> Tile 02 already exists and is valid, skipping download.
    -> Tile 03 already exists and is valid, skipping download.
    -> Tile 04 already exists and is valid, skipping download.
    -> Tile 05 already exists and is valid, skipping download.
    -> Tile 06 already exists and is valid, skipping download.
    -> Tile 07 already exists and is valid, skipping download.
    -> Tile 08 already exists and is valid, skipping download.
    -> Tile 09 already exists and is valid, skipping download.
    -> Requesting FABDEM for bbox: W=64.9966, E=67.9966, S=37.1843, N=40.1843 (attempt 1/3)
       ❌ Error during download attempt 1: HTTP 400 from OpenTopography.
URL: https://portal.opentopography.org/API/globaldem?demtype=FABDEM&west=64.99663505